<a href="https://colab.research.google.com/github/athfizh/PemMes26_05_Hafizh/blob/main/JS11/JS11-01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

# **Convolutional Neural Network (CNN) - Lab 1**

---

---

# **Praktikum Persiapan - Upload & Ekstrak Dataset**

---

In [ ]:
import os
import zipfile

# Cek apakah folder dataset sudah ada
if not os.path.exists('dataset'):
    # Cek apakah file dataset.zip tersedia di direktori kerja
    if os.path.exists('dataset.zip'):
        print("Mengekstrak dataset.zip...")
        with zipfile.ZipFile('dataset.zip', 'r') as zip_ref:
            zip_ref.extractall('.')
        print("Ekstraksi dataset.zip berhasil!")
    else:
        # Jika di Google Colab dan file belum diunggah
        try:
            from google.colab import files
            print("Silakan upload file dataset.zip:")
            uploaded = files.upload()
            for fn in uploaded.keys():
                if fn.endswith('.zip'):
                    with zipfile.ZipFile(fn, 'r') as zip_ref:
                        zip_ref.extractall('.')
                    print(f"Ekstraksi {fn} berhasil!")
        except ImportError:
            print("Jalankan script ini di Google Colab atau pastikan folder 'dataset/' tersedia di direktori kerja.")
else:
    print("Folder 'dataset/' sudah tersedia dan siap digunakan.")

Mengekstrak dataset.zip...
Ekstraksi dataset.zip berhasil!


---

# **Langkah 1 - Import Library**

---

> Import library utama yang digunakan: `tensorflow` sebagai framework deep learning dan `ImageDataGenerator` untuk manipulasi dan pembacaan citra secara langsung dari struktur direktori.

In [ ]:
import tensorflow as tf
# Di Google Colab (TensorFlow 2.16+ / Keras 3), ImageDataGenerator berada di tensorflow.keras.preprocessing.image
try:
    from tensorflow.keras.preprocessing.image import ImageDataGenerator
except (ImportError, AttributeError):
    try:
        from keras.preprocessing.image import ImageDataGenerator
    except (ImportError, AttributeError):
        from keras.src.legacy.preprocessing.image import ImageDataGenerator

---

# **Langkah 2 - Pra Pengolahan Data**

---

> **Langkah 2.1. Pra Pengolahan Data Training**
>
> Pada tahap ini kita akan sedikit melakukan manipulasi pada citra yang digunakan. Manipulasi yang dilakukan diantaranya adalah normalisasi nilai piksel (`rescale`), koreksi kemiringan (`shear_range`), pembesaran (`zoom_range`), dan pembalikan horizontal (`horizontal_flip`).

In [ ]:
train_datagen = ImageDataGenerator(rescale = 1./255,
                                   shear_range = 0.2,
                                   zoom_range = 0.2,
                                   horizontal_flip = True)
training_set = train_datagen.flow_from_directory('dataset/training_set',
                                                 target_size = (64, 64),
                                                 batch_size = 32,
                                                 class_mode = 'binary')

Found 8000 images belonging to 2 classes.


> `ImageDataGenerator` untuk data training berhasil dikonfigurasi. Citra diubah ukurannya menjadi dimensi seragam 64×64 piksel, dibaca per batch ukuran 32, dan label diatur sebagai biner (`cats=0`, `dogs=1`).

> **Langkah 2.2. Pra Pengolahan Data Testing**
>
> Untuk data testing, manipulasi citra hanya dilakukan pada normalisasi nilai piksel (`rescale = 1./255`) tanpa augmentasi geometris, agar pengujian mencerminkan evaluasi objektif pada citra asli.

In [ ]:
test_datagen = ImageDataGenerator(rescale = 1./255)
test_set = test_datagen.flow_from_directory('dataset/test_set',
                                            target_size = (64, 64),
                                            batch_size = 32,
                                            class_mode = 'binary')

Found 2000 images belonging to 2 classes.


---

# **Langkah 3 - Pembuatan Model CNN**

---

> **Langkah 3.1. - Inisiasi Model CNN**
>
> Inisiasi model jaringan saraf tiruan konvolusional menggunakan model Sequential dari TensorFlow Keras.

In [ ]:
cnn = tf.keras.models.Sequential()

> Objek model Sequential bernama `cnn` berhasil diinisialisasi sebagai wadah untuk menambahkan layer-layer konvolusi, pooling, dan fully connected secara berurutan.

> **Langkah 3.2. - Pembuatan Layer Konvolusi 1**
>
> Tambahkan layer konvolusi pertama dengan 32 filter/kernel berukuran 3×3, fungsi aktivasi ReLU, dan input shape [64, 64, 3] sesuai ukuran citra RGB 64×64 piksel.

In [ ]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu', input_shape=[64, 64, 3]))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


> **Langkah 3.3. - Pembuatan Layer Pooling 1**
>
> Tambahkan layer Max Pooling dengan pool size 2×2 dan stride 2 untuk mereduksi dimensi spasial citra sebesar 50% sekaligus mempertahankan fitur-fitur paling dominan.

In [ ]:
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

> Layer `MaxPool2D` pertama berhasil ditambahkan untuk proses downsampling representasi fitur dan mengurangi jumlah parameter komputasi.

> **Langkah 3.4. - Pembuatan Layer Konvolusi 2 dan Pooling 2**
>
> Tambahkan layer konvolusi kedua dengan 32 filter ukuran 3×3 dan aktivasi ReLU, diikuti layer Max Pooling kedua (pool size 2×2, stride 2) untuk mengekstrak representasi fitur yang lebih abstrak.

In [ ]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

> Blok konvolusi dan pooling kedua berhasil ditambahkan. Kombinasi ini memungkinkan model mendeteksi fitur yang lebih kompleks seperti tekstur dan bentuk bagian tubuh objek.

> **Langkah 3.5. - Flattening**
>
> Lakukan proses flattening untuk mengubah matriks fitur 2D/3D hasil pooling menjadi vektor 1 dimensi sebagai persiapan input ke Fully Connected (Dense) Layer.

In [ ]:
cnn.add(tf.keras.layers.Flatten())

> Layer `Flatten` berhasil meratakan seluruh peta fitur menjadi vektor 1D beruntun.

> **Langkah 3.6. - Fully Connected Layer 1 (Input)**
>
> Tambahkan hidden layer Fully Connected (Dense) pertama dengan 128 neuron/unit dan fungsi aktivasi ReLU untuk mempelajari kombinasi non-linear dari fitur-fitur yang telah diekstrak.

In [ ]:
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))

> Layer `Dense` dengan 128 unit ReLU berhasil ditambahkan sebagai layer pembelajaran pola klasifikasi.

> **Langkah 3.7. - Fully Connected Layer 2 (Output)**
>
> Tambahkan output layer dengan 1 neuron dan fungsi aktivasi Sigmoid untuk menghasilkan nilai probabilitas biner antara 0 dan 1 (0 untuk kucing, 1 untuk anjing).

In [ ]:
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

> Layer output `Dense` 1 neuron Sigmoid selesai ditambahkan. Nilai mendekati 0 mengindikasikan kelas kucing dan mendekati 1 mengindikasikan kelas anjing.

> **Langkah 3.8. - Compile Model CNN**
>
> Kompilasi model dengan optimizer `adam`, fungsi loss `binary_crossentropy` (karena klasifikasi biner 2 kelas: kucing dan anjing), dan metrik evaluasi `accuracy`.

In [ ]:
cnn.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

> Model `cnn` berhasil dikompilasi dan siap dilatih menggunakan data pelatihan yang telah disiapkan.

---

# **Langkah 4 - Fit CNN**

---

> Latih model CNN menggunakan data training `training_set` dengan validasi data `test_set` selama 25 epoch.

In [ ]:
cnn.fit(x = training_set, validation_data = test_set, epochs = 25)

Epoch 1/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 79s 309ms/step - accuracy: 0.5913 - loss: 0.6608 - val_accuracy: 0.6745 - val_loss: 0.6140
Epoch 2/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 77s 309ms/step - accuracy: 0.6871 - loss: 0.5951 - val_accuracy: 0.7220 - val_loss: 0.5507
Epoch 3/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 79s 316ms/step - accuracy: 0.7060 - loss: 0.5636 - val_accuracy: 0.7240 - val_loss: 0.5431
Epoch 4/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 74s 297ms/step - accuracy: 0.7415 - loss: 0.5230 - val_accuracy: 0.7395 - val_loss: 0.5425
Epoch 5/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 74s 297ms/step - accuracy: 0.7524 - loss: 0.5012 - val_accuracy: 0.7625 - val_loss: 0.4988
Epoch 6/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 79s 315ms/step - accuracy: 0.7563 - loss: 0.4909 - val_accuracy: 0.7685 - val_loss: 0.4810
Epoch 7/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 74s 296ms/step - accuracy: 0.7703 - loss: 0.4706 - val_accuracy: 0.7750 - val_loss: 0.4725
Epoch 8/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 75s 300ms/step - accuracy: 0.7862 - loss: 0

---

# **Langkah 5 - Prediksi dengan 1 Citra**

---

In [ ]:
import numpy as np
try:
    from tensorflow.keras.preprocessing import image
except (ImportError, AttributeError):
    try:
        from keras.preprocessing import image
    except (ImportError, AttributeError):
        from keras.src.legacy.preprocessing import image

test_image = image.load_img('dataset/single_prediction/cat_or_dog_1.jpg', target_size = (64, 64))
test_image = image.img_to_array(test_image)
test_image = np.expand_dims(test_image, axis = 0)
result = cnn.predict(test_image)
print("Indeks kelas:", training_set.class_indices)
print(f"Nilai probabilitas prediksi: {result[0][0]:.4f}")

# Output aktivasi sigmoid bernilai antara 0 (cat) dan 1 (dog)
if result[0][0] >= 0.5:
    prediction = 'dog'
else:
    prediction = 'cat'

print("Hasil Prediksi:", prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step
Indeks kelas: {'cats': 0, 'dogs': 1}
Nilai probabilitas prediksi: 1.0000
Hasil Prediksi: dog


> Citra berhasil dimuat dan diprediksi oleh model CNN. Model mencocokkan hasil aktivasi sigmoid dengan kelas (`dog` jika 1 atau mendekati 1, `cat` jika 0 atau mendekati 0) sesuai pemetaan indeks `training_set.class_indices`.